# Poverty, Income Inequality, and Health Outcomes Across Malaysian States
## Final Report

**Course:** COSC 301  
**Author:** Aliff Iman bin Abdul Razak
**Student Number:** 58423609   
**Date:** April 2026  

---

## Abstract

This report investigates socioeconomic inequality and public health infrastructure across all 16 Malaysian states and federal territories. Using household income, poverty, Gini coefficient, and health facility data drawn from OpenDOSM, the Ministry of Health Malaysia, and the World Bank, the study addresses four research questions spanning cross-sectional profiles, inequality-health correlations, recent trends (2019–2024), and a five-decade historical arc (1970–2024). A composite Socioeconomic Deprivation Index (SDI) is constructed to rank states on combined economic and health disadvantage.

Key findings: the income ratio between the richest and poorest state is 2.7×; Sabah, Sarawak, and Kelantan carry the heaviest poverty burden; higher-income states have *fewer* public hospital beds (Spearman ρ = −0.53, *p* = 0.041), driven by private sector substitution; median household income rose in all 15 non-administrative states between 2019 and 2024; and national poverty has fallen from ≈53% (1970) to ≈6% (2024). The SDI identifies Sabah, Sarawak, and Kelantan as the most deprived states, a ranking that is robust to alternative weighting schemes.


## Table of Contents

1. [Introduction & Research Questions](#1-introduction)
2. [Data Sources & Pipeline](#2-data)
3. [Methodology](#3-methodology)
4. [Findings](#4-findings)
   - 4.1 [Income & Poverty Landscape (2022)](#41-income-poverty)
   - 4.2 [Health Infrastructure Per Capita (2022)](#42-health)
   - 4.3 [Inequality vs Health Infrastructure](#43-inequality-health)
   - 4.4 [Recent Trends: 2019–2024](#44-trends)
   - 4.5 [Long-run History: 1970–2024](#45-longrun)
   - 4.6 [Socioeconomic Deprivation Index](#46-sdi)
5. [Conclusions & Policy Implications](#5-conclusions)
6. [Limitations](#6-limitations)
7. [Appendix: Data Dictionary](#7-appendix)


---
<a id='1-introduction'></a>
## 1. Introduction & Research Questions

Malaysia has achieved rapid economic development over the past five decades, but growth has not been uniform across its 16 states and federal territories. Persistent gaps in household income and poverty rates particularly between Peninsular and East Malaysia, raise questions about whether rising prosperity has been accompanied by equitable access to public services such as healthcare.

This project builds a reproducible data pipeline to answer the following research questions using 2022 as the primary reference year (the most complete Household Income and Expenditure Survey wave):

| # | Research Question |
|---|---|
| **Q1** | Which states have the highest and lowest household incomes, and where is the poverty burden most acute? |
| **Q2** | Which states are best and worst served by public health infrastructure per capita? |
| **Q3** | Does income inequality predict poorer public health infrastructure at the state level? |
| **Q4** | How did income and poverty change across states between 2019 and 2024? |
| **Q5** | How has national income and poverty evolved since 1970? |

An additional **SDI analysis** combines the economic and health dimensions into a single Socioeconomic Deprivation Index and tests the robustness of the resulting rankings.


---
<a id='2-data'></a>
## 2. Data Sources & Pipeline

### 2.1 Sources

| Source | Dataset | Coverage |
|---|---|---|
| OpenDOSM | `hh_income_parlimen.csv` | Constituency income, 2019/2022/2024 |
| OpenDOSM | `hh_poverty_parlimen.csv` | Constituency poverty, 2019/2022/2024 |
| OpenDOSM | `hh_income_state.csv` | State income history, 1970–2022 |
| OpenDOSM | `hh_poverty_state.csv` | State poverty history, 1970–2022 |
| OpenDOSM | `hh_inequality_state.csv` | Official household Gini, 1974–2022 |
| OpenDOSM | `population_state.csv` | Annual state population, 1970–2025 |
| OpenDOSM | `cpi_2d_annual.csv` | National annual CPI by MCOICOP division, 1960–2025 |
| MoH Malaysia | `moh_facilities.csv` | Health facility snapshot (~2023/2024) |
| MoH Malaysia | `moh_beds.csv` | Hospital bed snapshot (~2023/2024) |
| World Bank | Multiple indicators | National life expectancy, infant mortality, health expenditure |

All raw files are downloaded automatically by [01_acquire_data.ipynb](01_acquire_data.ipynb) / `scripts/acquire.py`.

### 2.2 Data Pipeline

```
OpenDOSM / MoH / World Bank APIs
        │
        ▼
data/raw/           ← 01_acquire_data.ipynb
        │
        ▼
data/clean/         ← 02_etl.ipynb  (validate → transform → export)
        │
        ├── combined_state.csv    (319 rows, 1970–2024)
        ├── health_state.csv      (48 rows, 3 years × 16 states)
        ├── sdi_scores.csv        (16 rows, 2022 reference year)
        ├── cpi_national.csv      (66 rows, 1960–2025)
        └── dashboard_data.xlsx   (4 sheets)
        │
        ▼
malaysia_project.db ← SQLite (same 4 tables)
```

| Notebook | Purpose |
|---|---|
| [01_acquire_data.ipynb](01_acquire_data.ipynb) | Download and cache all raw files |
| [02_etl.ipynb](02_etl.ipynb) | Raw data audit, transformation, export |
| [03_eda.ipynb](03_eda.ipynb) | Exploratory analysis — Q1–Q5 |
| [04_sdi_analysis.ipynb](04_sdi_analysis.ipynb) | SDI construction and sensitivity checks |

Full ETL design decisions are documented in [docs/methodology.md](../docs/methodology.md). To query the outputs interactively, see [docs/sqlite_quickstart.sql](../docs/sqlite_quickstart.sql).

### 2.3 Data Quality Notes

The raw audit ([02_etl.ipynb §1](02_etl.ipynb)) found:

- No duplicate records in any source file after normalising state name variants.
- `moh_beds` had a single negative value (`beds_nonicu = -4`), clamped to 0.
- `moh_facilities` had 12 duplicate facility name × state keys, retained as separate facilities.
- `hh_poverty_state` had sparse coverage of `poverty_relative` and `poverty_hardcore` before 1989; only `poverty_absolute` is used throughout.
- National CPI (`cpi_2d_annual`) covers 1960–2025 and is joined to `combined_state` on `year`, providing a deflator for every survey year in the historical series.

---
<a id='3-methodology'></a>
## 3. Methodology

> Full methodology is documented in [docs/methodology.md](../docs/methodology.md).

### 3.1 Unit of Analysis

The primary unit is the Malaysian **state** (16 units including three federal territories). Raw constituency and facility data are aggregated to state level before analysis.

W.P. Putrajaya is excluded from the main analysis set as a purpose-built administrative enclave; it is included in SDI sensitivity checks.

### 3.2 Socioeconomic Aggregation

State-level income and poverty figures for 2019, 2022, and 2024 are derived as **unweighted means across parliamentary constituencies** within each state. This approach is used because population weights per constituency are not available in the DOSM source.

The `gini` column in the recent series measures **inter-constituency dispersion** of mean incomes, not household-level inequality. Official DOSM household Gini values (where available from `hh_inequality_state.csv`) replace this proxy.

### 3.3 Health Per-Capita Metrics

Public health infrastructure is measured as:
- **Beds per 1,000 population** (`beds_per_1000`)
- **Facilities per 100,000 population** (`facilities_per_100k`)

The MoH source files carry no year dimension; the same counts are applied across all analysis years and only the population denominator varies.

### 3.4 Statistical Methods

| Method | Use |
|---|---|
| Descriptive statistics & rankings | Q1, Q2: income/poverty/health league tables |
| Spearman ρ | Q3: inequality vs health (robust to small n and outliers) |
| Pearson r | Q3: supplementary linear correlation |
| OLS regression | Q3: `beds_per_1000 ~ income_mean` (exploratory) |
| Two-sample t-test | Q1: East vs West income comparison |
| Percentage change | Q4: income and poverty trend 2019–2024 |

Significance threshold: α = 0.05. With n = 16, only large effects are detectable; non-significant results should not be interpreted as evidence of no relationship. See [docs/methodology.md — Statistical Methods](../docs/methodology.md#statistical-methods) for full details.

### 3.5 SDI Construction

Each of five indicators is min-max normalised to [0, 1], with inverted indicators subtracted from 1 so that **1 = most deprived** on every component:

| Indicator | Direction | Weight |
|---|---|---|
| Poverty rate | Higher → more deprived | **25%** |
| Median income (inverted) | Lower → more deprived | **25%** |
| Gini coefficient | Higher → more deprived | **20%** |
| Beds per 1,000 (inverted) | Fewer → more deprived | **15%** |
| Facilities per 100k (inverted) | Fewer → more deprived | **15%** |

The **double deprivation** flag is set when a state is above the median on *both* the economic sub-score (poverty + Gini + low income) and the health sub-score (bed + facility shortage).

Two sensitivity checks are run: (A) equal weights (20% each) and (B) re-normalisation excluding federal territories. Full workings in [04_sdi_analysis.ipynb §1](04_sdi_analysis.ipynb).


---
<a id='4-findings'></a>
## 4. Findings


<a id='41-income-poverty'></a>
### 4.1 Income & Poverty Landscape (Q1 — 2022)

> Full analysis: [03_eda.ipynb §1 - Income & Poverty Landscape](03_eda.ipynb)

#### Income Rankings

| Rank | State | Median Income (RM) | Mean Income (RM) |
|---|---|---|---|
| 1 | W.P. Kuala Lumpur | 10,271 | 12,613 |
| 2 | W.P. Putrajaya | 10,056 | 12,163 |
| 3 | Selangor | 9,146 | 11,159 |
| 4 | W.P. Labuan | 6,904 | 8,418 |
| 5 | Pulau Pinang | 6,426 | 8,012 |
| … | … | … | … |
| 14 | Sabah | 4,446 | 5,601 |
| 15 | Sarawak | 4,316 | 5,374 |
| 16 | Kelantan | 3,698 | 4,793 |

The **richest-to-poorest median income ratio is 2.7×** (W.P. Kuala Lumpur vs Kelantan). An independent t-test comparing East and West Malaysia mean incomes does **not** reach significance, indicating that within-region dispersion drives overall inequality more than the East–West geographic divide.

#### Poverty Rankings

| Rank | State | Poverty Rate (%) |
|---|---|---|
| 1 (highest) | Sabah | 21.8 |
| 2 | Sarawak | 13.7 |
| 3 | Kelantan | 13.5 |
| 4 | Perak | 9.1 |
| 5 | Kedah | 8.3 |
| … | … | … |
| 16 (lowest) | W.P. Putrajaya | 0.1 |

Sabah, Sarawak, and Kelantan each record poverty rates **more than double the unweighted national average**. These three states account for the most acute deprivation across both income and poverty dimensions.

![Income ranking by state](../figures/fig1_income_ranking.png)

*Figure 1. Median household income by state (2022). East Malaysia states are highlighted in orange.*     

![Poverty ranking by state](../figures/fig2_poverty_ranking.png)

*Figure 2. Absolute poverty rate by state (2022). States with rates above the median are highlighted.*     


<a id='42-health'></a>
### 4.2 Health Infrastructure Per Capita (Q2 - 2022)

> Full analysis: [03_eda.ipynb §2 — Health Infrastructure Per Capita](03_eda.ipynb)

#### Hospital Beds per 1,000 Population

| Rank | State | Beds / 1,000 | Facilities / 100k | Hospitals |
|---|---|---|---|---|
| 1 (best) | W.P. Putrajaya | 6.75 | 5.1 | 2 |
| 2 | Perlis | 1.77 | 14.1 | 1 |
| 3 | Sarawak | 1.53 | 11.9 | 23 |
| 4 | Perak | 1.39 | 14.6 | 16 |
| 5 | Kelantan | 1.37 | 14.8 | 10 |
| … | … | … | … | … |
| 14 | Pulau Pinang | 0.87 | 6.4 | 6 |
| 15 | Sabah | 0.72 | 10.4 | 25 |
| 16 (fewest) | Selangor | 0.48 | 3.5 | 18 |

**W.P. Putrajaya** is a structural outlier: two federal hospitals serve a small administrative population, producing an artificially high ratio. **Selangor** appears worst by public bed count, but this is explained by its large private hospital sector which is not captured in MoH data.

Peninsular West states (Selangor, Pulau Pinang, Johor) record the lowest public bed ratios but have the highest incomes and most developed private healthcare markets. Rural East Malaysian states and Kelantan have more public beds per capita but worse income and poverty outcomes — capturing a different form of disadvantage.

![Beds ranking](../figures/fig3_beds_ranking.png)

*Figure 3. Public hospital beds per 1,000 population by state (2022).*


<a id='43-inequality-health'></a>
### 4.3 Inequality vs Health Infrastructure (Q3)

> Full analysis: [03_eda.ipynb §3 - Inequality & Health Correlation](03_eda.ipynb)

#### Correlation Results

| Relationship | Spearman ρ | p-value | Interpretation |
|---|---|---|---|
| Mean income vs beds/1000 | −0.53 | 0.041 | Significant negative — richer states have *fewer* public beds |
| Gini vs facilities/100k | −0.55 | 0.035 | Significant negative — lower dispersion → more facilities |
| Poverty rate vs beds/1000 | Not significant | > 0.05 | No significant linear relationship |

The **negative income–beds correlation** is counter-intuitive at first glance but is explained by private sector substitution: high-income urban states (Selangor, Pulau Pinang, W.P. Kuala Lumpur) host large private hospital markets that absorb demand and reduce pressure on public facilities. MoH data captures only the public sector.

The **Gini–facilities correlation** (ρ = −0.55) is more interpretable in policy terms: states with lower inter-constituency income dispersion tend to have better public facility coverage, suggesting that geographic income clustering and infrastructure inequality move together.

The OLS regression (`beds_per_1000 ~ income_mean`, n = 16) is exploratory only — with 16 data points it cannot support causal inference.

![Income vs beds scatter](../figures/fig4_income_vs_beds.png)

*Figure 4. Income vs public hospital beds per capita. The negative slope reflects private sector substitution in high-income states.*

![Poverty vs facilities scatter](../figures/fig5_poverty_vs_facilities.png)

*Figure 5. Poverty rate vs public facilities per 100k population.*

![Correlation heatmap](../figures/fig7_correlation_heatmap.png)

*Figure 6. Spearman correlation matrix across all socioeconomic and health variables (2022).*


<a id='44-trends'></a>
### 4.4 Recent Trends: 2019–2024 (Q4)

> Full analysis: [03_eda.ipynb §4 - Trends 2019–2024](03_eda.ipynb)

#### Median Income Change

| State | 2019 (RM) | 2022 (RM) | 2024 (RM) | % Change 2019–2024 |
|---|---|---|---|---|
| Selangor | ~7,121 | ~9,146 | ~9,127 | **+28.2%** |
| Sabah | ~3,993 | ~4,446 | ~4,948 | +23.9% |
| Kelantan | ~3,037 | ~3,698 | ~3,567 | +17.5% |
| W.P. Kuala Lumpur | ~10,055 | ~10,271 | ~10,327 | **+2.7%** |

Median income **rose in all 15 non-administrative states** between 2019 and 2024. The 2020–2021 pandemic gap is visible in the data (no HIES survey was conducted in those years).

**W.P. Kuala Lumpur** recorded the smallest percentage gain (+2.7%) but from the highest starting base. **Selangor** grew most (+28.2%). The gap between the richest and poorest states has not materially narrowed in absolute terms.

#### Poverty Change

Poverty fell in **12 of 15 states** between 2019 and 2024.

- **Sabah** recorded the largest absolute reduction (−2.4 pp) but remains the highest-poverty state at 19.3% in 2024.
- Three states saw poverty increase slightly between 2019 and 2024 (provisional 2024 data).

![Income trends](../figures/fig6_income_trends.png)

*Figure 7. Median household income trajectories by state, 2019–2024. Selected states highlighted.*


<a id='45-longrun'></a>
### 4.5 Long-run History: 1970–2024 (Q5)

> Full analysis: [03_eda.ipynb §5 - Long-run History](03_eda.ipynb)

#### National Averages (Unweighted State Mean)

| Year | Mean Income (RM nominal) | Poverty Rate (%) |
|---|---|---|
| 1970 | ~213 | ~52.9 |
| 1984 | ~1,099 | ~20.7 |
| 1999 | ~2,893 | ~12.4 |
| 2012 | ~5,750 | ~3.9 |
| 2022 | ~7,254 | ~6.6 |
| 2024 | ~8,222 | ~6.0 |

Key headline figures:
- National mean income grew **approximately 31× in nominal terms** from 1970 to 2024.
- Absolute poverty fell from **≈52.9%** (1970) to **≈6.0%** (2024), one of the most sustained poverty-reduction trajectories in Southeast Asia.
- The 2020–2021 period shows a data gap (no HIES surveys during the COVID-19 pandemic).
- States that started with the worst outcomes — **Perlis, Kelantan, and Terengganu** — have converged substantially toward the national average over this period, though gaps remain.

> **Caveat:** All income figures are nominal (not inflation-adjusted). Real income growth is considerably lower than the 31× headline figure.

![Historical income](../figures/fig8_historical_income.png)

*Figure 8. State mean household income 1970–2024 (nominal RM). Selected states highlighted; grey lines show all other states.*


<a id='46-sdi'></a>
### 4.6 Socioeconomic Deprivation Index (SDI — 2022)

> Full analysis and sensitivity workings: [04_sdi_analysis.ipynb](04_sdi_analysis.ipynb)

#### Final Rankings

| Rank | State | SDI Score | Double Deprived |
|---|---|---|---|
| 1 (most deprived) | Sabah | 0.883 | ★ |
| 2 | Sarawak | 0.787 | ★ |
| 3 | Kelantan | 0.694 | — |
| 4 | Negeri Sembilan | 0.630 | ★ |
| 5 | Perak | 0.617 | — |
| 6 | Kedah | 0.609 | — |
| 7 | Johor | 0.596 | ★ |
| 8 | Pulau Pinang | 0.555 | — |
| 9 | Pahang | 0.532 | — |
| 10 | Selangor | 0.512 | — |
| 11 | Perlis | 0.497 | — |
| 12 | Terengganu | 0.495 | — |
| 13 | Melaka | 0.490 | — |
| 14 | W.P. Kuala Lumpur | 0.443 | — |
| 15 | W.P. Labuan | 0.345 | — |
| 16 (least deprived) | W.P. Putrajaya | 0.139 | — |

★ **Double deprived**: above the median on both the economic sub-score (poverty + Gini + low income) and the health sub-score (bed + facility shortage). These four states - Sabah, Sarawak, Negeri Sembilan, and Johor - face compounded disadvantage and represent the highest-priority targets for integrated policy intervention.

#### Normalised Component Scores (2022)

| State | Poverty (25%) | Income (25%) | Gini (20%) | Beds (15%) | Facilities (15%) |
|---|---|---|---|---|---|
| Sabah | 1.000 | 0.886 | 0.888 | 0.962 | 0.599 |
| Sarawak | 0.628 | 0.906 | 1.000 | 0.833 | 0.521 |
| Kelantan | 0.616 | 1.000 | 0.528 | 0.858 | 0.369 |
| Selangor | 0.087 | 0.171 | 0.769 | **1.000** | 0.957 |
| W.P. Kuala Lumpur | 0.060 | **0.000** | 0.728 | 0.880 | **1.000** |
| W.P. Putrajaya | **0.000** | 0.033 | **0.000** | **0.000** | 0.873 |

*0 = least deprived on that component; 1 = most deprived.*

![SDI ranking](../figures/fig9_sdi_ranking.png)

*Figure 9. State Deprivation Index scores (2022). States marked with ^ are double deprived.*

![SDI component breakdown](../figures/fig10_sdi_components.png)

*Figure 10. Weighted SDI contribution by component. ★ marks double-deprived states.*

![Double deprivation quadrant](../figures/fig11_double_deprivation.png)

*Figure 11. Double deprivation quadrant — states above both medians (shaded region) face compounded economic and health disadvantage.*

#### Sensitivity Analysis

> Detailed rank shift tables: [04_sdi_analysis.ipynb §4 — Sensitivity Analysis](04_sdi_analysis.ipynb)

Two alternative specifications were tested:

**A. Equal weights (20% each) vs proposed differential weights:**
- Top 3 and bottom 3 ranks are unchanged.
- The largest shift is W.P. Kuala Lumpur moving from rank 14 to rank 10 under equal weights, because its low-income component (25% → 20%) carries less weight.
- Mid-table states shift ±1–2 positions.

**B. Excluding federal territories (re-normalising 13 states only):**
- Top 3 and bottom 3 state rankings are unchanged.
- Mid-table shifts are at most ±2 positions.
- Federal territories anchor the min-max extremes; removing them slightly re-distributes mid-range scores but does not change the identity of the most or least deprived states.

**Conclusion:** The double-deprivation flag is stable across both sensitivity checks — the same four states appear in the upper-right quadrant regardless of weighting scheme, confirming that their compounded disadvantage is a genuine feature of the data rather than an artefact of weight choice.

![SDI sensitivity](../figures/fig12_sdi_sensitivity.png)

*Figure 12. SDI sensitivity analysis: (A) proposed vs equal weights; (B) with vs without federal territories.*


---
<a id='5-conclusions'></a>
## 5. Conclusions & Policy Implications

### 5.1 Summary of Findings

| Research Question | Key Finding |
|---|---|
| Q1 - Income & poverty | 2.7× income gap between richest and poorest state. Sabah, Sarawak, and Kelantan carry poverty rates more than double the national average. |
| Q2 - Health infrastructure | Public bed and facility coverage is highest in less-urbanised states; lowest in high-income urban states that rely on private sector. |
| Q3 - Inequality vs health | Richer states have fewer public beds (ρ = −0.53, *p* = 0.041) - explained by private sector substitution. States with lower geographic income dispersion have more public facilities (ρ = −0.55, *p* = 0.035). |
| Q4 - Recent trends | Median income rose in all 15 non-administrative states 2019–2024. Poverty fell in 12 of 15 states. Sabah improved most in absolute terms but remains the highest-poverty state. |
| Q5 - Long-run history | Poverty fell from ≈53% (1970) to ≈6% (2024). Nominal income grew ≈31×. Convergence is visible but gaps persist. |
| SDI | Sabah, Sarawak, Kelantan are the three most deprived states. Four states - Sabah, Sarawak, Negeri Sembilan, Johor — are doubly deprived on both economic and health sub-scores. Rankings are robust under alternative weights. |

### 5.2 Policy Implications

**1. Prioritise doubly-deprived states for integrated programmes.**  
Sabah and Sarawak face simultaneous high poverty, low incomes, high income dispersion, and relative public health shortfalls. Programmes that address only one dimension will be insufficient. Bundled interventions — combining social transfer programmes with health infrastructure investment — are most likely to reduce SDI scores in these states.

**2. Interpret public health metrics in the context of the private sector.**  
The paradox of high-income states (Selangor, W.P. Kuala Lumpur) appearing poorly served by public beds is explained by private healthcare substitution. Policy monitoring should incorporate total (public + private) healthcare capacity, particularly for planning purposes in rapidly urbanising states.

**3. The East–West narrative obscures within-region heterogeneity.**  
East Malaysia as a category spans very different conditions: Sarawak has better public health infrastructure than Sabah, and W.P. Labuan performs well on most indicators. Targeted policy design should be at the state level, not the regional level.

**4. Sustain the long-run poverty reduction trajectory.**  
The 1970–2024 poverty reduction record is strong. The remaining hard-core poor are concentrated in East Malaysia and rural Kelantan — likely the hardest populations to reach through broad-based growth alone, and requiring direct targeted interventions.

**5. Treat 2024 figures as provisional.**  
DOSM labels the 2024 HIES data as provisional. The upward income trend should be interpreted directionally; exact magnitudes will update when final figures are released.


---
<a id='6-limitations'></a>
## 6. Limitations

> Full caveats with impact assessments: [docs/caveats.md](../docs/caveats.md)

| # | Limitation | Impact | Mitigation |
|---|---|---|---|
| 1 | MoH health data is a point-in-time snapshot (~2023/24), not time-series | Cross-sectional ranking valid; cannot track health trends | Treated as baseline; replicated across analysis years |
| 2 | Population denominators use 2020 census (not 2019 or 2024 estimates) | Slight under/over-statement of per-capita rates | Small relative to state-to-state variation |
| 3 | Gini measures inter-constituency dispersion, not household inequality | Not directly comparable to published DOSM Gini | Documented explicitly; labelled accordingly |
| 4 | Constituency aggregation uses unweighted means | Small rural constituencies equal-weighted with large urban ones | Population weights unavailable in source data |
| 5 | 2024 HIES data is provisional | Trend endpoint may be revised | Cross-sectional analysis uses 2022 |
| 6 | n = 16 states gives low statistical power | Non-significant results may reflect power, not absence of relationship | Both Pearson and Spearman reported; OLS treated as exploratory |
| 7 | No 2020–2021 HIES surveys (COVID-19) | Pandemic shock and recovery cannot be tracked continuously | Annotated in trend charts |
| 8 | Historical and constituency-derived income series use different methods | Values for the same state-year may differ across series | Historical series used for long-run work; constituency series for recent cross-sections |
| 9 | National CPI assumes a uniform consumption basket across all states | Does not capture urban–rural price differences or low-income spending patterns | Noted in EDA §7; real income figures treated as indicative |
| 10 | Federal territories have atypical profiles | Direct comparison with states imperfect | Tagged via `territory_type`; excluded from some sensitivity checks |

---
<a id='7-appendix'></a>
## 7. Appendix: Data Dictionary

> Full data dictionary: [docs/data_dictionary.md](../docs/data_dictionary.md)  
> SQL query examples: [docs/sqlite_quickstart.sql](../docs/sqlite_quickstart.sql)

### `combined_state.csv` (319 rows)

| Column | Type | Description |
|---|---|---|
| `state_code` | str | ISO-style 3-letter state code |
| `state` | str | Canonical state name |
| `year` | int | Survey year |
| `territory_type` | str | `state`, `capital`, `admin`, or `island_ft` |
| `income_mean` | float | Mean of constituency mean incomes (RM) |
| `income_median` | float | Mean of constituency median incomes (RM) |
| `poverty_absolute` | float | Mean constituency absolute poverty rate (%) |
| `gini` | float | Inter-constituency Gini (or official DOSM Gini where available) |
| `cpi_overall` | float | National headline CPI for that year (base 2010 = 100) |

### `health_state.csv` (48 rows)

| Column | Type | Description |
|---|---|---|
| `state` | str | Canonical state name |
| `year` | int | Analysis year (denominator varies; infrastructure counts are static) |
| `beds_per_1000` | float | Public hospital beds per 1,000 population |
| `facilities_per_100k` | float | Public health facilities per 100,000 population |
| `hospital_count` | int | Number of public hospitals |
| `facility_count` | int | Total public health facilities |
| `beds_total` | int | Total public hospital beds (non-ICU + ICU) |

### `sdi_scores.csv` (16 rows)

| Column | Type | Description |
|---|---|---|
| `state` | str | Canonical state name |
| `sdi_score` | float | Composite SDI (0 = least deprived, 1 = most deprived) |
| `sdi_rank` | int | Rank (1 = most deprived) |
| `double_deprivation` | bool | True if above median on both economic and health sub-scores |
| `n_poverty` | float | Normalised poverty component |
| `n_gini` | float | Normalised Gini component |
| `n_income` | float | Normalised income component (inverted) |
| `n_beds` | float | Normalised beds component (inverted) |
| `n_facilities` | float | Normalised facilities component (inverted) |
| `sdi_rank_eq` | int | Rank under equal weights (20% each) |
| `sdi_rank_no_terr` | float | Rank when federal territories are excluded from normalisation |

### `cpi_national.csv` (66 rows)

| Column | Type | Description |
|---|---|---|
| `year` | int | Year (1960–2025) |
| `cpi_overall` | float | National headline CPI (`division = 'overall'`, base year 2010 = 100) |

---

### SDI Weights Reference

```python
SDI_WEIGHTS = {
    "n_poverty":      0.25,
    "n_gini":         0.20,
    "n_income":       0.25,
    "n_beds":         0.15,
    "n_facilities":   0.15,
}
```

### State Codes

| State | Code | | State | Code |
|---|---|---|---|---|
| Johor | JHR | | Pulau Pinang | PNG |
| Kedah | KDH | | Sabah | SBH |
| Kelantan | KTN | | Sarawak | SWK |
| Melaka | MLK | | Selangor | SGR |
| Negeri Sembilan | NSN | | Terengganu | TRG |
| Pahang | PHG | | W.P. Kuala Lumpur | KUL |
| Perak | PRK | | W.P. Labuan | LBN |
| Perlis | PLS | | W.P. Putrajaya | PJY |